# Load Environment

In [1]:
%load_ext dotenv
%dotenv

# Import Required Modeuls

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables import RunnableParallel

# Chat Template

In [3]:
chat_template_books = ChatPromptTemplate.from_template(
    """
    Suggest three of the best intermediate-level {programming language} books.
    Answer only by listing the books.
    """
)
chat_template_projects = ChatPromptTemplate.from_template(
    """
    Suggest three interesting {programming language} project suitable for intermidiate-level programmers.
    Answer only by listing the projects.
    """
)

chat_template_time = ChatPromptTemplate.from_template(
    """
    I'm an intermediate level programmer.
    Consider the following literature:
    {books}
    Also, consider the following projects:
    {projects}
    Roughly how much time would it take me to complete the literature and the projects?
    """
)

# OpenAI model

In [4]:
chat = ChatOpenAI(model = "gpt-4",
                  model_kwargs = {"seed": 365},
                  temperature = 0,
                  max_tokens = 500
                 )

C:\ProgramData\anaconda3\envs\langchain_env\lib\site-packages\IPython\core\interactiveshell.py:3519: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


# Output Parser

In [5]:
string_parser = StrOutputParser()

# Piping Chain

In [6]:
chain_books = chat_template_books | chat | string_parser
chain_projects = chat_template_projects | chat | string_parser

# Runnable Parallel

In [7]:
chain_parallel = RunnableParallel({"books": chain_books, 
                                   "projects": chain_projects})

In [8]:
chain_parallel.invoke({"programming language" : "python"})

{'books': '1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin',
 'projects': '1. Building a Weather Application using API: This project involves fetching data from a weather API and displaying it in a user-friendly manner. \n\n2. Developing a Content Aggregator: This project involves creating a platform that gathers content from various sources across the internet and presents it in one location.\n\n3. Creating a Personal Finance Tracker: This project involves building a tool that tracks income, expenses, and provides budgeting features.'}

# Piping Runnables With Other Runnables

In [9]:
chain_time = (RunnableParallel({"books" : chain_books,
                                "projects": chain_projects})
              | chat_template_time
              | chat
              | string_parser
             )

In [10]:
print(chain_time.invoke({"programming language" : "python"}))

The time it takes to complete the literature and the projects can vary greatly depending on several factors such as your reading speed, comprehension level, the complexity of the projects, and the amount of time you can dedicate each day. 

For the books, if you read for about 2 hours a day:

1. "Fluent Python: Clear, Concise, and Effective Programming" - This book is around 800 pages. If you read at an average speed of 30 pages per hour, it would take you approximately 27 days to finish.

2. "Python Cookbook: Recipes for Mastering Python 3" - This book is around 700 pages. Using the same reading speed, it would take you approximately 23 days to finish.

3. "Effective Python: 90 Specific Ways to Write Better Python" - This book is around 230 pages. It would take you approximately 8 days to finish.

For the projects, it's harder to estimate without knowing your proficiency and the complexity of the projects you have in mind. However, as a rough estimate:

1. Building a Weather Forecast 

# Graph Runnables

In [11]:
chain_time.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

# Runnable Lambda

In [12]:
sum_runnable = RunnableLambda(lambda x : sum(x))

In [13]:
sum_runnable.invoke([1,2,5])

8

In [15]:
Square_runnable = RunnableLambda(lambda x : x**2)

In [16]:
Square_runnable.invoke(8)

64

# Chain RunnableLambda

In [17]:
chain = sum_runnable | Square_runnable

In [18]:
chain.invoke([1,2,4])

49